# Rename from pretty names to original ones

In [ ]:
project_name = "{{ project_name }}"

In [ ]:
from pyprojroot import here
crowns_path = here(f"{project_name}_crowns.gpkg")
grid_path = here(f"{project_name}_grid.gpkg")
dir_cor_pretty = here("Corrected_pretty_names")
dir_cor = here("Corrected_pcs")
dir_sel = here("Propagation/selected_trees")

## Make map

an old tree name is made of a `<project-prefix>_<cell-id>_<tree-id>` and gets mapped to a pretty name of `<pretty-cell-name>_<tree-id>` for example:

```
'Lama_02_seg.laz' --> 'FIS01-02_345910_8059490_02_seg.laz'
```

since there are new trees we need to reconstruct this mapping.

The first step is to load the grid mapping

In [ ]:
import geopandas as gpd
from fastcore.xtras import *
from pathlib import Path

In [ ]:
grid = gpd.read_file(grid_path)
grid.head(5)

,cell_id,grid_x,grid_y,pretty_name,geometry
0,345910_8059490,345910,8059490,Lama,"POLYGON ((345920 8059490, 345920 8059500, 3459..."
1,345910_8059500,345910,8059500,Mada,"POLYGON ((345920 8059500, 345920 8059510, 3459..."
2,345910_8059510,345910,8059510,Obus,"POLYGON ((345920 8059510, 345920 8059520, 3459..."
3,345920_8059480,345920,8059480,Aata,"POLYGON ((345930 8059480, 345930 8059490, 3459..."
4,345920_8059490,345920,8059490,Kali,"POLYGON ((345930 8059490, 345930 8059500, 3459..."


get the prefix from the crowns file (that has the mapping for the existing trees)

In [ ]:
crowns = gpd.read_file(crowns_path)

In [ ]:
crowns.head(5)

,GIS_ID,ID_number,area,area.1,cell_id,grid_x,grid_y,grid_x_norm,grid_y_norm,hilbert_dist,suffix,pretty_name,segmented,notes,is_dead,is_removed,is_new,geometry
0,FIS01-02_345910_8059490_02,345910_8059490_02,3.651367,3.651367,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,02,Lama_02,True,None,False,NaN,NaN,"POLYGON ((345910.853 8059491.292, 345910.858 8..."
1,FIS01-02_345910_8059490_05,345910_8059490_05,16.017580,16.017580,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,05,Lama_05,True,None,False,NaN,NaN,"POLYGON ((345908.036 8059493.668, 345908.036 8..."
2,FIS01-02_345910_8059490_06,345910_8059490_06,4.727412,4.727412,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,06,Lama_06,True,None,False,NaN,NaN,"POLYGON ((345912.998 8059497.032, 345913.008 8..."
3,FIS01-02_345910_8059490_07,345910_8059490_07,10.067097,10.067097,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,07,Lama_07,True,None,False,NaN,NaN,"POLYGON ((345913.142 8059489.798, 345913.19 80..."
4,FIS01-02_345910_8059490_09,345910_8059490_09,3.543199,3.543199,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,09,Lama_09,True,None,False,NaN,NaN,"POLYGON ((345914.558 8059495.864, 345914.568 8..."


In [ ]:
prefix = crowns['GIS_ID'][0].split("_")[0]
prefix

'FIS01-02'

In [ ]:
grid_pretty2old = {
    row.pretty_name: f"{prefix}_{row.cell_id}"
    for _, row in grid.iterrows()
}
grid_pretty2old

{'Lama': 'FIS01-02_345910_8059490',
 'Mada': 'FIS01-02_345910_8059500',
 'Obus': 'FIS01-02_345910_8059510',
 'Aata': 'FIS01-02_345920_8059480',
 'Kali': 'FIS01-02_345920_8059490',
 'Jaya': 'FIS01-02_345920_8059500',
 'Naia': 'FIS01-02_345920_8059510',
 'Odax': 'FIS01-02_345920_8059520',
 'Bako': 'FIS01-02_345930_8059470',
 'Ecto': 'FIS01-02_345930_8059480',
 'Faba': 'FIS01-02_345930_8059490',
 'Iago': 'FIS01-02_345930_8059500',
 'Psoa': 'FIS01-02_345930_8059510',
 'Okea': 'FIS01-02_345930_8059520',
 'Omus': 'FIS01-02_345930_8059530',
 'Zuma': 'FIS01-02_345930_8059540',
 'Ceto': 'FIS01-02_345940_8059470',
 'Daku': 'FIS01-02_345940_8059480',
 'Gaga': 'FIS01-02_345940_8059490',
 'Hada': 'FIS01-02_345940_8059500',
 'Paha': 'FIS01-02_345940_8059510',
 'Oxya': 'FIS01-02_345940_8059520',
 'Owra': 'FIS01-02_345940_8059530',
 'Zora': 'FIS01-02_345940_8059540',
 'Zeus': 'FIS01-02_345940_8059550',
 'Zaus': 'FIS01-02_345940_8059560',
 'Ugni': 'FIS01-02_345950_8059490',
 'Ucla': 'FIS01-02_345950_80

In [ ]:
def get_old_name(pretty_tree: str):
    cell_name = pretty_tree.split("_")[0]
    new_name = pretty_tree.replace(cell_name, grid_pretty2old[cell_name])
    return new_name

In [ ]:
get_old_name("Aata_02_seg.laz")

'FIS01-02_345920_8059480_02_seg.laz'

In [ ]:
pretty2old = {
    pretty_tree: get_old_name(pretty_tree.name) for pretty_tree in dir_cor_pretty.ls()
}
pretty2old

{Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_02_seg.laz'): 'FIS01-02_345920_8059480_02_seg.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_05_seg.laz'): 'FIS01-02_345920_8059480_05_seg.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_08_seg.laz'): 'FIS01-02_345920_8059480_08_seg.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_14_seg.laz'): 'FIS01-02_345920_8059480_14_seg.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_15_seg.laz'): 'FIS01-02_345920_8059480_15_seg.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_16_seg.laz'): 'FIS01-02_345920_8059480_16_seg.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_17_seg.laz'): 'FIS01-02_345920_8059480_17_seg.laz',
 Path('c:/Use

## Suffix

I am using the `_seg` suffix, while I should use the `_cor` suffix

In [ ]:
pretty2old = {
    path: old_name.replace("_seg", "_cor") for path, old_name in pretty2old.items()
}
pretty2old

{Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_02_seg.laz'): 'FIS01-02_345920_8059480_02_cor.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_05_seg.laz'): 'FIS01-02_345920_8059480_05_cor.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_08_seg.laz'): 'FIS01-02_345920_8059480_08_cor.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_14_seg.laz'): 'FIS01-02_345920_8059480_14_cor.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_15_seg.laz'): 'FIS01-02_345920_8059480_15_cor.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_16_seg.laz'): 'FIS01-02_345920_8059480_16_cor.laz',
 Path('c:/Users/smassaro/Documents/segmentation/Mount_fisher/Corrected_pretty_names/Aata_17_seg.laz'): 'FIS01-02_345920_8059480_17_cor.laz',
 Path('c:/Use

## Copy and rename

In [ ]:
import shutil
from pathlib import Path


def copy_rename_files(
    path_map: dict, # pretty path to old name
    target_folder: Path,
):
    for pretty_path, old_name in path_map.items():
        new_name = target_folder / old_name
        shutil.copy(pretty_path, new_name)
        print(f"{pretty_path.name} -> {new_name.name}")

In [ ]:
dir_cor.mkdir(exist_ok=True)

In [ ]:
len(pretty2old)

472

In [ ]:
copy_rename_files(pretty2old, dir_cor)

Aata_02_seg.laz -> FIS01-02_345920_8059480_02_cor.laz
Aata_05_seg.laz -> FIS01-02_345920_8059480_05_cor.laz
Aata_08_seg.laz -> FIS01-02_345920_8059480_08_cor.laz
Aata_14_seg.laz -> FIS01-02_345920_8059480_14_cor.laz
Aata_15_seg.laz -> FIS01-02_345920_8059480_15_cor.laz
Aata_16_seg.laz -> FIS01-02_345920_8059480_16_cor.laz
Aata_17_seg.laz -> FIS01-02_345920_8059480_17_cor.laz
Bako_12_seg.laz -> FIS01-02_345930_8059470_12_cor.laz
Bako_15_seg.laz -> FIS01-02_345930_8059470_15_cor.laz
Bako_25_seg.laz -> FIS01-02_345930_8059470_25_cor.laz
Ceto_06_seg.laz -> FIS01-02_345940_8059470_06_cor.laz
Ceto_14_seg.laz -> FIS01-02_345940_8059470_14_cor.laz
Ceto_19_seg.laz -> FIS01-02_345940_8059470_19_cor.laz
Daku_01_seg.laz -> FIS01-02_345940_8059480_01_cor.laz
Daku_08_seg.laz -> FIS01-02_345940_8059480_08_cor.laz
Daku_09_seg.laz -> FIS01-02_345940_8059480_09_cor.laz
Daku_10_seg.laz -> FIS01-02_345940_8059480_10_cor.laz
Daku_11_seg.laz -> FIS01-02_345940_8059480_11_cor.laz
Daku_12_seg.laz -> FIS01-02_

## Updated crowns vector

Need to make sure that the crowns vector file contains two columsn with this format

```
GIS_ID	ID_number
WHY01-01_322330_8191020_18	322330_8191020_18
WHY01-01_322330_8191020_31	322330_8191020_31
WHY01-01_322330_8191020_35d	322330_8191020_35d
WHY01-01_322340_8190980_17	322340_8190980_17
```

I need to **overwrite** because of the new trees

In [ ]:
crowns['GIS_ID'] = crowns['pretty_name'].map(get_old_name)
crowns['ID_number'] = crowns['GIS_ID'].str.replace(f"{prefix}_", "")

In [ ]:
crowns.head(5)

,GIS_ID,ID_number,area,area.1,cell_id,grid_x,grid_y,grid_x_norm,grid_y_norm,hilbert_dist,suffix,pretty_name,segmented,notes,is_dead,is_removed,is_new,geometry
0,FIS01-02_345910_8059490_02,345910_8059490_02,3.651367,3.651367,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,02,Lama_02,True,None,False,NaN,NaN,"POLYGON ((345910.853 8059491.292, 345910.858 8..."
1,FIS01-02_345910_8059490_05,345910_8059490_05,16.017580,16.017580,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,05,Lama_05,True,None,False,NaN,NaN,"POLYGON ((345908.036 8059493.668, 345908.036 8..."
2,FIS01-02_345910_8059490_06,345910_8059490_06,4.727412,4.727412,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,06,Lama_06,True,None,False,NaN,NaN,"POLYGON ((345912.998 8059497.032, 345913.008 8..."
3,FIS01-02_345910_8059490_07,345910_8059490_07,10.067097,10.067097,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,07,Lama_07,True,None,False,NaN,NaN,"POLYGON ((345913.142 8059489.798, 345913.19 80..."
4,FIS01-02_345910_8059490_09,345910_8059490_09,3.543199,3.543199,345910_8059490,345910.0,8059490.0,0.0,20.0,944.0,09,Lama_09,True,None,False,NaN,NaN,"POLYGON ((345914.558 8059495.864, 345914.568 8..."


In [ ]:
crowns.to_file(crowns_path, driver="GPKG", layer = crowns_path.stem)